# Regresión logística en 1D

Este cuaderno presenta un ejemplo simple de **clasificación binaria** con una sola entrada por muestra.

Se trabaja con los datos del Live Script original y se interpreta el problema como una clasificación en una dimensión.

## 1. Datos de entrenamiento

Cada muestra tiene una sola característica $x$ y una etiqueta binaria $y$.

$
X = \begin{bmatrix}1\\2\\3\\4\end{bmatrix}
$

$
y = \begin{bmatrix}1\\1\\0\\0\end{bmatrix}
$

Interpretación rápida:

- las muestras con $x=1$ y $x=2$ pertenecen a la clase 1,
- las muestras con $x=3$ y $x=4$ pertenecen a la clase 0.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Semilla fija para obtener siempre el mismo resultado
np.random.seed(7)

# Datos del problema
X = np.array([[1.0],
              [2.0],
              [3.0],
              [4.0]])

y = np.array([[1],
              [1],
              [0],
              [0]])

# Tabla simple para revisar los datos
datos = pd.DataFrame({
    'x': X.ravel(),
    'y': y.ravel()
})
datos

## 2. Visualización de los datos

Como el problema es 1D, todas las muestras se dibujan sobre el eje horizontal.

La coordenada vertical se fija en cero solo para facilitar la visualización.

In [ ]:
# Separamos las muestras por clase
x_clase_0 = X[y.ravel() == 0]
x_clase_1 = X[y.ravel() == 1]

plt.figure(figsize=(8, 4))
plt.scatter(x_clase_0, np.zeros(len(x_clase_0)), label='Clase 0', s=90)
plt.scatter(x_clase_1, np.zeros(len(x_clase_1)), label='Clase 1', s=90)
plt.xlabel('x')
plt.ylabel('posición auxiliar')
plt.title('Datos de entrenamiento')
plt.xlim(0, 5)
plt.ylim(-1, 1)
plt.grid(True)
plt.legend()
plt.show()

## 3. Modelo de regresión logística

La regresión logística calcula primero una combinación lineal:

$$
z = \theta_0 + \theta_1 x
$$

Luego transforma ese valor en una probabilidad mediante la función sigmoide:

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

La hipótesis del modelo es:

$$
h_\theta(x) = \sigma(\theta_0 + \theta_1 x)
$$

Si la probabilidad es alta, el modelo se inclina por la clase 1. Si es baja, se inclina por la clase 0.

### Función de costo

Durante el entrenamiento se busca minimizar el error promedio del modelo:

$$
J(\theta) = -\frac{1}{m}\sum_{i=1}^{m}\left[y^{(i)}\log\left(h_\theta(x^{(i)})\right) + (1-y^{(i)})\log\left(1-h_\theta(x^{(i)})\right)\right]
$$

En este cuaderno se usa **descenso por gradiente** para actualizar los parámetros.

In [ ]:
# Número de muestras y número de características
m, n = X.shape

# Añadimos una columna de unos para representar el sesgo θ0
X_bias = np.hstack([np.ones((m, 1)), X])

# Inicialización aleatoria de los parámetros
theta = 2 * np.random.rand(n + 1, 1) - 1

# Hiperparámetros del entrenamiento
alpha = 0.01
iterations = 5000

# Función sigmoide
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Lista para guardar el valor de la función de costo
cost_history = []

# Entrenamiento con descenso por gradiente
for _ in range(iterations):
    # Paso hacia adelante: cálculo de probabilidades
    h = sigmoid(X_bias @ theta)

    # Gradiente de la función de costo
    gradient = (X_bias.T @ (h - y)) / m

    # Actualización de parámetros
    theta = theta - alpha * gradient

    # Cálculo del costo para observar el aprendizaje
    eps = 1e-12
    cost = -(1 / m) * np.sum(y * np.log(h + eps) + (1 - y) * np.log(1 - h + eps))
    cost_history.append(float(cost))

print('Parámetros aprendidos:')
print(theta)

## 4. Predicción

Con los parámetros ya entrenados, el modelo calcula una probabilidad para cada muestra.

La regla de decisión es:

$$
\hat{y} =
\begin{cases}
1 & \text{si } h_\theta(x) \geq 0.5 \\0 & \text{si } h_\theta(x) < 0.5
\end{cases}
$$

In [ ]:
# Probabilidades estimadas
probabilities = sigmoid(X_bias @ theta)

# Conversión de probabilidad a clase
predictions = (probabilities >= 0.5).astype(int)

# Exactitud del modelo
accuracy = np.mean(predictions == y) * 100

resultados = pd.DataFrame({
    'x': X.ravel(),
    'y_real': y.ravel(),
    'probabilidad_clase_1': probabilities.ravel(),
    'y_predicha': predictions.ravel()
})

print(f'Exactitud del modelo: {accuracy:.2f}%')
resultados

## 5. Evolución del aprendizaje

La siguiente gráfica muestra cómo cambia la función de costo durante las iteraciones.

Si la curva desciende, significa que el modelo está ajustando mejor sus parámetros.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(cost_history)
plt.xlabel('Iteración')
plt.ylabel('Costo')
plt.title('Evolución de la función de costo')
plt.grid(True)
plt.show()

## 6. Frontera de decisión

En una sola dimensión, la frontera de decisión es un punto sobre el eje $x$.

Se obtiene cuando la combinación lineal vale cero:

$$
\theta_0 + \theta_1 x = 0
$$

Despejando $x$:

$$
x_{\text{frontera}} = -\frac{\theta_0}{\theta_1}
$$

Ese valor divide la región donde el modelo predice clase 1 de la región donde predice clase 0.

In [ ]:
# Cálculo de la frontera de decisión en 1D
x_boundary = -theta[0, 0] / theta[1, 0]

plt.figure(figsize=(8, 4))
plt.scatter(x_clase_0, np.zeros(len(x_clase_0)), label='Clase 0', s=90)
plt.scatter(x_clase_1, np.zeros(len(x_clase_1)), label='Clase 1', s=90)
plt.axvline(x=x_boundary, linewidth=2, label='Frontera de decisión')
plt.xlabel('x')
plt.ylabel('posición auxiliar')
plt.title('Datos y frontera de decisión')
plt.xlim(0, 5)
plt.ylim(-1, 1)
plt.grid(True)
plt.legend()
plt.show()

print(f'Frontera de decisión: x = {x_boundary:.4f}')

---

## 7. Conclusión

En este ejemplo, la regresión logística:

- recibe un valor de entrada $x$,
- calcula una probabilidad de pertenecer a la clase 1,
- toma una decisión usando el umbral de 0.5.

Este flujo es la base de muchos problemas de clasificación binaria.